In [9]:
#Public Debt and Fiscal Sustainability in India: Complete Econometric Analysis
#Based on Provided Dataset (1991-2024) 

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [11]:
# Setting visualisation style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

# ============================================================================
# SECTION 1: LOAD AND PREPARE DATA
# ============================================================================

# Load your dataset
data = {
    'Year': [1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000,
             2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
             2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
             2021, 2022, 2023, 2024],
    'Debt_GDP': [68.1, 67.5, 66.8, 65.9, 65.3, 66.2, 67.1, 69.0, 70.5, 72.2,
                 74.0, 75.3, 74.8, 73.2, 71.5, 69.0, 66.5, 68.1, 72.5, 70.8,
                 69.4, 68.7, 67.8, 66.9, 66.1, 65.5, 65.2, 66.0, 69.0, 89.0,
                 87.5, 85.2, 84.3, 83.5],
    'GDP_Growth': [1.1, 5.5, 4.8, 6.7, 7.6, 7.5, 4.0, 6.2, 8.8, 4.1,
                   4.8, 3.8, 8.1, 7.9, 9.3, 9.2, 9.8, 3.9, 8.5, 10.3,
                   6.6, 5.5, 6.4, 7.4, 8.0, 8.3, 6.8, 6.5, 4.0, -6.6,
                   8.9, 7.2, 7.0, 6.9],
    'Interest_Rate': [11.5, 11.0, 10.8, 10.6, 10.5, 10.8, 11.0, 11.3, 11.6, 11.8,
                      11.7, 11.4, 10.9, 10.5, 10.1, 9.8, 9.4, 8.9, 8.4, 8.2,
                      8.3, 8.4, 8.6, 8.5, 8.1, 7.8, 7.6, 7.7, 7.9, 6.8,
                      6.5, 7.1, 7.3, 7.4],
    'Primary_Deficit': [3.3, 2.8, 2.6, 2.5, 2.4, 2.7, 3.0, 3.3, 3.5, 3.7,
                        3.6, 3.4, 2.8, 2.3, 2.0, 1.7, 1.4, 2.5, 3.8, 3.4,
                        3.1, 3.0, 2.8, 2.6, 2.5, 2.4, 2.3, 2.4, 3.2, 7.5,
                        6.7, 6.0, 5.8, 5.5]
}

df = pd.DataFrame(data)

# Calculate Primary Balance (negative of Primary Deficit)
df['Primary_Balance'] = -df['Primary_Deficit']

print("="*80)
print("DATA LOADED SUCCESSFULLY")
print("="*80)
print(f"Period: {df['Year'].min()} - {df['Year'].max()}")
print(f"Number of observations: {len(df)}")


DATA LOADED SUCCESSFULLY
Period: 1991 - 2024
Number of observations: 34


In [12]:
# ============================================================================
# SECTION 2: DESCRIPTIVE STATISTICS (Already run, just confirming)
# ============================================================================

print("\n" + "="*80)
print("SECTION 2: DESCRIPTIVE STATISTICS (Summary)")
print("="*80)

print("\nDescriptive Statistics:")
desc_stats = df[['Debt_GDP', 'GDP_Growth', 'Interest_Rate', 'Primary_Deficit', 'Primary_Balance']].describe()
print(desc_stats)

# Calculate r - g differential
df['r_minus_g'] = df['Interest_Rate'] - df['GDP_Growth']

print("\nr - g Differential Statistics:")
print(df['r_minus_g'].describe())
print(f"Years with r < g (favorable): {(df['r_minus_g'] < 0).sum()}/{len(df)}")
print(f"Years with r > g (unfavorable): {(df['r_minus_g'] > 0).sum()}/{len(df)}")



SECTION 2: DESCRIPTIVE STATISTICS (Summary)

Descriptive Statistics:
        Debt_GDP  GDP_Growth  Interest_Rate  Primary_Deficit  Primary_Balance
count  34.000000   34.000000      34.000000        34.000000        34.000000
mean   71.423529    6.317647       9.358824         3.308824        -3.308824
std     6.740743    3.062032       1.665172         1.399863         1.399863
min    65.200000   -6.600000       6.500000         1.400000        -7.500000
25%    66.825000    4.975000       7.950000         2.500000        -3.475000
50%    69.000000    6.850000       9.150000         2.900000        -2.900000
75%    73.025000    8.075000      10.875000         3.475000        -2.500000
max    89.000000   10.300000      11.800000         7.500000        -1.400000

r - g Differential Statistics:
count    34.000000
mean      3.041176
std       3.515000
min      -2.400000
25%       0.525000
50%       2.700000
75%       5.075000
max      13.400000
Name: r_minus_g, dtype: float64
Years with r

In [13]:
# ============================================================================
# SECTION 3: UNIT ROOT TESTS (Augmented Dickey-Fuller) - CORRECTED
# ============================================================================

from statsmodels.tsa.stattools import adfuller

print("\n" + "="*80)
print("SECTION 3: UNIT ROOT TESTS (Augmented Dickey-Fuller)")
print("="*80)

def adf_test(series, variable_name, max_lags=4):
    """Perform ADF test and print results"""
    result = adfuller(series, autolag='AIC', maxlag=max_lags)
    print(f"\n{variable_name}:")
    print(f"  ADF Statistic: {result[0]:.6f}")
    print(f"  p-value: {result[1]:.6f}")
    print(f"  Critical Values:")
    for key, value in result[4].items():
        print(f"    {key}: {value:.3f}")
    if result[1] < 0.05:
        print(f"  Conclusion: Stationary (reject H0 at 5% significance)")
        return True, result
    else:
        print(f"  Conclusion: Non-stationary (fail to reject H0)")
        return False, result

# Test level variables
print("\n--- Testing Level Variables (I(0)) ---")
stationary_at_level = {}
for var in ['Debt_GDP', 'GDP_Growth', 'Interest_Rate', 'Primary_Deficit', 'Primary_Balance']:
    stationary, _ = adf_test(df[var], var)
    stationary_at_level[var] = stationary

# Test first differences
print("\n--- Testing First Differences (I(1)) ---")
df_diff = df[['Debt_GDP', 'GDP_Growth', 'Interest_Rate', 'Primary_Deficit', 'Primary_Balance']].diff().dropna()
stationary_at_diff = {}
for var in df_diff.columns:
    stationary, _ = adf_test(df_diff[var], f"Δ{var}")
    stationary_at_diff[var] = stationary

# Summary of stationarity
print("\n--- Summary of Unit Root Tests ---")
print("Variable        | Level (I(0)) | First Diff (I(1)) | Order")
print("-" * 60)
for var in ['Debt_GDP', 'GDP_Growth', 'Interest_Rate', 'Primary_Deficit', 'Primary_Balance']:
    level_stat = "Stationary" if stationary_at_level[var] else "Non-stationary"
    diff_stat = "Stationary" if stationary_at_diff[var] else "Non-stationary"
    order = "I(0)" if stationary_at_level[var] else "I(1)"
    print(f"{var:16} | {level_stat:12} | {diff_stat:15} | {order}")



SECTION 3: UNIT ROOT TESTS (Augmented Dickey-Fuller)

--- Testing Level Variables (I(0)) ---

Debt_GDP:
  ADF Statistic: -1.200146
  p-value: 0.673533
  Critical Values:
    1%: -3.646
    5%: -2.954
    10%: -2.616
  Conclusion: Non-stationary (fail to reject H0)

GDP_Growth:
  ADF Statistic: -5.310512
  p-value: 0.000005
  Critical Values:
    1%: -3.646
    5%: -2.954
    10%: -2.616
  Conclusion: Stationary (reject H0 at 5% significance)

Interest_Rate:
  ADF Statistic: -0.780939
  p-value: 0.824657
  Critical Values:
    1%: -3.654
    5%: -2.957
    10%: -2.618
  Conclusion: Non-stationary (fail to reject H0)

Primary_Deficit:
  ADF Statistic: -1.501462
  p-value: 0.532788
  Critical Values:
    1%: -3.646
    5%: -2.954
    10%: -2.616
  Conclusion: Non-stationary (fail to reject H0)

Primary_Balance:
  ADF Statistic: -1.501462
  p-value: 0.532788
  Critical Values:
    1%: -3.646
    5%: -2.954
    10%: -2.616
  Conclusion: Non-stationary (fail to reject H0)

--- Testing First

In [14]:
# ============================================================================
# SECTION 4: COINTEGRATION ANALYSIS (Johansen Test) - CORRECTED
# ============================================================================

from statsmodels.tsa.vector_ar.vecm import coint_johansen

print("\n" + "="*80)
print("SECTION 4: JOHANSEN COINTEGRATION TEST")
print("="*80)

# Prepare data for cointegration
coint_data = df[['Debt_GDP', 'GDP_Growth', 'Interest_Rate', 'Primary_Deficit']].values

# Johansen test with lag length 1 (corrected attribute access)
print("\n--- Lag Length: 1 ---")
johansen_result = coint_johansen(coint_data, det_order=0, k_ar_diff=1)

print("\nJohansen Cointegration Test Results:")
print(f"Eigenvalues: {johansen_result.eig}")
print(f"Number of cointegrating vectors: {johansen_result.eig.shape[0]}")

print("\nTrace Test (r <= i):")
for i in range(len(johansen_result.eig)):
    print(f"  r <= {i}: Trace Statistic = {johansen_result.lr1[i]:.4f}")
    print(f"         Critical Value (90%): {johansen_result.cvt[i, 0]:.4f}")
    print(f"         Critical Value (95%): {johansen_result.cvt[i, 1]:.4f}")
    print(f"         Critical Value (99%): {johansen_result.cvt[i, 2]:.4f}")
    if johansen_result.lr1[i] > johansen_result.cvt[i, 1]:
        print(f"         → Reject H0 at 5%: At least {i+1} cointegrating relationship(s)")
    elif johansen_result.lr1[i] > johansen_result.cvt[i, 0]:
        print(f"         → Reject H0 at 10%: At least {i+1} cointegrating relationship(s)")
    else:
        print(f"         → Cannot reject H0: Less than {i+1} cointegrating relationships")
    print()

print("\nMaximum Eigenvalue Test (r = i):")
for i in range(len(johansen_result.eig)):
    print(f"  r = {i}: Max Eigen Statistic = {johansen_result.lr2[i]:.4f}")
    print(f"         Critical Value (95%): {johansen_result.cvm[i, 1]:.4f}")
    if johansen_result.lr2[i] > johansen_result.cvm[i, 1]:
        print(f"         → Reject H0 at 5%: Cointegrating relationship exists")
    else:
        print(f"         → Cannot reject H0: No cointegrating relationship")
    print()

print("\nNormalized Cointegrating Vector (first vector):")
for idx, var in enumerate(['Debt_GDP', 'GDP_Growth', 'Interest_Rate', 'Primary_Deficit']):
    print(f"  {var}: {johansen_result.evec[idx, 0]:.6f}")



SECTION 4: JOHANSEN COINTEGRATION TEST

--- Lag Length: 1 ---

Johansen Cointegration Test Results:
Eigenvalues: [0.65599844 0.37778641 0.17882219 0.0204722 ]
Number of cointegrating vectors: 4

Trace Test (r <= i):
  r <= 0: Trace Statistic = 56.2970
         Critical Value (90%): 44.4929
         Critical Value (95%): 47.8545
         Critical Value (99%): 54.6815
         → Reject H0 at 5%: At least 1 cointegrating relationship(s)

  r <= 1: Trace Statistic = 22.1495
         Critical Value (90%): 27.0669
         Critical Value (95%): 29.7961
         Critical Value (99%): 35.4628
         → Cannot reject H0: Less than 2 cointegrating relationships

  r <= 2: Trace Statistic = 6.9664
         Critical Value (90%): 13.4294
         Critical Value (95%): 15.4943
         Critical Value (99%): 19.9349
         → Cannot reject H0: Less than 3 cointegrating relationships

  r <= 3: Trace Statistic = 0.6619
         Critical Value (90%): 2.7055
         Critical Value (95%): 3.8415
    

In [15]:
# ============================================================================
# SECTION 5: REGRESSION MODELS
# ============================================================================

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_white, acorr_breusch_godfrey
from statsmodels.stats.stattools import durbin_watson

print("\n" + "="*80)
print("SECTION 5: REGRESSION MODELS")
print("="*80)

# ============================================================================
# MODEL 1: Debt Sustainability Model
# ============================================================================

print("\n" + "-"*60)
print("MODEL 1: Debt Sustainability Model")
print("Debt_t = β0 + β1*GDP_Growth_t + β2*Interest_Rate_t + β3*Primary_Deficit_t + ε_t")
print("-"*60)

# Prepare data
X1 = df[['GDP_Growth', 'Interest_Rate', 'Primary_Deficit']]
X1 = sm.add_constant(X1)
y1 = df['Debt_GDP']

# Estimate model
model1 = sm.OLS(y1, X1).fit()

print("\nModel Summary:")
print(model1.summary())

print("\n--- Key Results ---")
print(f"R-squared: {model1.rsquared:.4f}")
print(f"Adjusted R-squared: {model1.rsquared_adj:.4f}")
print(f"F-statistic: {model1.fvalue:.4f}")
print(f"F-statistic p-value: {model1.f_pvalue:.6f}")

print("\nCoefficients:")
for var in model1.params.index:
    coef = model1.params[var]
    pval = model1.pvalues[var]
    se = model1.bse[var]
    tval = model1.tvalues[var]
    print(f"  {var}: {coef:.4f} (t={tval:.4f}, p={pval:.4f})")
    if var != 'const' and pval < 0.05:
        print(f"    → Statistically significant at 5% level")
    elif var != 'const' and pval < 0.10:
        print(f"    → Statistically significant at 10% level")



SECTION 5: REGRESSION MODELS

------------------------------------------------------------
MODEL 1: Debt Sustainability Model
Debt_t = β0 + β1*GDP_Growth_t + β2*Interest_Rate_t + β3*Primary_Deficit_t + ε_t
------------------------------------------------------------

Model Summary:
                            OLS Regression Results                            
Dep. Variable:               Debt_GDP   R-squared:                       0.857
Model:                            OLS   Adj. R-squared:                  0.843
Method:                 Least Squares   F-statistic:                     60.11
Date:                Sun, 05 Apr 2026   Prob (F-statistic):           8.58e-13
Time:                        14:13:39   Log-Likelihood:                -79.507
No. Observations:                  34   AIC:                             167.0
Df Residuals:                      30   BIC:                             173.1
Df Model:                           3                                         
Covar

In [16]:
# ============================================================================
# MODEL 2: Bohn (1998) Sustainability Model
# ============================================================================

print("\n" + "-"*60)
print("MODEL 2: Bohn (1998) Sustainability Model")
print("PrimaryBalance_t = α + β*Debt_{t-1} + γ*OutputGap_t + ε_t")
print("-"*60)

# Create lagged debt variable
df['Debt_GDP_lag1'] = df['Debt_GDP'].shift(1)

# Calculate output gap using HP filter approximation
from statsmodels.tsa.filters.hp_filter import hpfilter
cycle, trend = hpfilter(df['GDP_Growth'], lamb=100)
df['Output_Gap'] = cycle

# Drop missing values
df_model2 = df.dropna(subset=['Primary_Balance', 'Debt_GDP_lag1', 'Output_Gap'])

# Prepare data
X2 = df_model2[['Debt_GDP_lag1', 'Output_Gap']]
X2 = sm.add_constant(X2)
y2 = df_model2['Primary_Balance']

# Estimate model
model2 = sm.OLS(y2, X2).fit()

print("\nModel Summary:")
print(model2.summary())

print("\n--- Key Results ---")
print(f"R-squared: {model2.rsquared:.4f}")
print(f"Adjusted R-squared: {model2.rsquared_adj:.4f}")
print(f"F-statistic: {model2.fvalue:.4f}")
print(f"F-statistic p-value: {model2.f_pvalue:.6f}")

print("\nCoefficients:")
for var in model2.params.index:
    coef = model2.params[var]
    pval = model2.pvalues[var]
    se = model2.bse[var]
    tval = model2.tvalues[var]
    print(f"  {var}: {coef:.4f} (t={tval:.4f}, p={pval:.4f})")

# Sustainability test
beta_debt = model2.params['Debt_GDP_lag1']
beta_pvalue = model2.pvalues['Debt_GDP_lag1']

print("\n--- SUSTAINABILITY TEST (Bohn Framework) ---")
if beta_debt > 0 and beta_pvalue < 0.05:
    print(f"✓ H1 ACCEPTED: Primary balance responds positively to lagged debt")
    print(f"  β = {beta_debt:.4f}, p-value = {beta_pvalue:.4f}")
    print(f"  Conclusion: India's public debt is fiscally sustainable")
elif beta_debt > 0 and beta_pvalue < 0.10:
    print(f"⚠ Partial evidence: β = {beta_debt:.4f} (p={beta_pvalue:.4f})")
    print(f"  Conclusion: Weak evidence of fiscal sustainability")
elif beta_debt > 0:
    print(f"⚠ β = {beta_debt:.4f} but not statistically significant (p={beta_pvalue:.4f})")
    print(f"  Conclusion: Positive response but statistically insignificant")
else:
    print(f"✗ H0 ACCEPTED: No positive response of primary balance to debt")
    print(f"  β = {beta_debt:.4f}, p-value = {beta_pvalue:.4f}")
    print(f"  Conclusion: India's public debt is NOT fiscally sustainable")



------------------------------------------------------------
MODEL 2: Bohn (1998) Sustainability Model
PrimaryBalance_t = α + β*Debt_{t-1} + γ*OutputGap_t + ε_t
------------------------------------------------------------

Model Summary:
                            OLS Regression Results                            
Dep. Variable:        Primary_Balance   R-squared:                       0.691
Model:                            OLS   Adj. R-squared:                  0.670
Method:                 Least Squares   F-statistic:                     33.49
Date:                Sun, 05 Apr 2026   Prob (F-statistic):           2.27e-08
Time:                        14:13:39   Log-Likelihood:                -38.565
No. Observations:                  33   AIC:                             83.13
Df Residuals:                      30   BIC:                             87.62
Df Model:                           2                                         
Covariance Type:            nonrobust             

In [17]:
# ============================================================================
# SECTION 6: DIAGNOSTIC TESTS
# ============================================================================

print("\n" + "="*80)
print("SECTION 6: DIAGNOSTIC TESTS")
print("="*80)

# 6.1 Durbin-Watson Test
print("\n6.1 Durbin-Watson Test (Autocorrelation):")
dw_stat_model2 = durbin_watson(model2.resid)
print(f"  Model 2 Durbin-Watson statistic: {dw_stat_model2:.4f}")
if dw_stat_model2 < 1.5:
    print("  → Positive autocorrelation detected")
elif dw_stat_model2 > 2.5:
    print("  → Negative autocorrelation detected")
else:
    print("  → No significant autocorrelation")

# 6.2 Breusch-Godfrey LM Test
print("\n6.2 Breusch-Godfrey LM Test (Serial Correlation):")
bg_test_model2 = acorr_breusch_godfrey(model2, nlags=2)
print(f"  LM statistic: {bg_test_model2[0]:.4f}, p-value: {bg_test_model2[1]:.4f}")
if bg_test_model2[1] < 0.05:
    print("  → Serial correlation present")
else:
    print("  → No significant serial correlation")

# 6.3 White Test for Heteroskedasticity
print("\n6.3 White Test (Heteroskedasticity):")
white_test_model2 = het_white(model2.resid, model2.model.exog)
print(f"  White LM statistic: {white_test_model2[0]:.4f}, p-value: {white_test_model2[1]:.4f}")
if white_test_model2[1] < 0.05:
    print("  → Heteroskedasticity present")
else:
    print("  → No significant heteroskedasticity")

# 6.4 Residual Normality Test
print("\n6.4 Residual Normality Test (Jarque-Bera):")
jb_stat_model2, jb_pvalue_model2 = stats.jarque_bera(model2.resid)
print(f"  Jarque-Bera: {jb_stat_model2:.4f}, p-value: {jb_pvalue_model2:.4f}")
if jb_pvalue_model2 < 0.05:
    print("  → Residuals are not normally distributed")
else:
    print("  → Residuals appear normally distributed")

# 6.5 Variance Inflation Factor (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("\n6.5 Variance Inflation Factor (Multicollinearity):")
X1_vif = X1.drop('const', axis=1)
vif_data = pd.DataFrame()
vif_data["Variable"] = X1_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X1_vif.values, i) for i in range(len(X1_vif.columns))]
print(vif_data)
if (vif_data["VIF"] > 10).any():
    print("  → High multicollinearity detected (VIF > 10)")
else:
    print("  → No severe multicollinearity")



SECTION 6: DIAGNOSTIC TESTS

6.1 Durbin-Watson Test (Autocorrelation):
  Model 2 Durbin-Watson statistic: 0.6727
  → Positive autocorrelation detected

6.2 Breusch-Godfrey LM Test (Serial Correlation):
  LM statistic: 15.3793, p-value: 0.0005
  → Serial correlation present

6.3 White Test (Heteroskedasticity):
  White LM statistic: 14.7619, p-value: 0.0114
  → Heteroskedasticity present

6.4 Residual Normality Test (Jarque-Bera):
  Jarque-Bera: 0.9503, p-value: 0.6218
  → Residuals appear normally distributed

6.5 Variance Inflation Factor (Multicollinearity):
          Variable       VIF
0       GDP_Growth  4.729266
1    Interest_Rate  8.652725
2  Primary_Deficit  4.406270
  → No severe multicollinearity


In [18]:
# ============================================================================
# SECTION 7: FORECASTING
# ============================================================================

print("\n" + "="*80)
print("SECTION 7: FORECASTING (2025-2029)")
print("="*80)

# Generate forecast for next 5 years
forecast_years = np.arange(2025, 2030)
forecast_data = pd.DataFrame({
    'Year': forecast_years,
    'GDP_Growth': [6.5, 6.3, 6.2, 6.0, 5.8],
    'Interest_Rate': [7.5, 7.6, 7.7, 7.8, 7.9],
    'Primary_Deficit': [5.3, 5.1, 5.0, 4.8, 4.7],
    'Output_Gap': [0.3, 0.2, 0.1, 0.0, -0.1]
})

# Add constant for Model 1
X_forecast1 = sm.add_constant(forecast_data[['GDP_Growth', 'Interest_Rate', 'Primary_Deficit']])
forecast_data['Debt_GDP_forecast'] = model1.predict(X_forecast1)

# For Model 2, use last actual debt as starting point
forecast_data['Debt_GDP_lag1'] = [df['Debt_GDP'].iloc[-1]] + list(forecast_data['Debt_GDP_forecast'].iloc[:-1])
X_forecast2 = sm.add_constant(forecast_data[['Debt_GDP_lag1', 'Output_Gap']])
forecast_data['Primary_Balance_forecast'] = model2.predict(X_forecast2)

print("\nForecast Results (2025-2029):")
print(forecast_data[['Year', 'Debt_GDP_forecast', 'Primary_Balance_forecast']].to_string(index=False))



SECTION 7: FORECASTING (2025-2029)

Forecast Results (2025-2029):
 Year  Debt_GDP_forecast  Primary_Balance_forecast
 2025          80.732208                 -5.381722
 2026          79.717206                 -4.934247
 2027          79.220152                 -4.786453
 2028          78.205149                 -4.727215
 2029          77.674030                 -4.579421


In [19]:
# ============================================================================
# SECTION 8: SUMMARY OF FINDINGS
# ============================================================================

print("\n" + "="*80)
print("SECTION 8: SUMMARY OF FINDINGS")
print("="*80)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    FISCAL SUSTAINABILITY ANALYSIS SUMMARY                   │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  1. DESCRIPTIVE ANALYSIS:                                                   │
│     • Average Debt-GDP ratio: {:.1f}% (Range: {:.1f}% - {:.1f}%)                │
│     • Average GDP Growth: {:.1f}% (Range: {:.1f}% - {:.1f}%)                   │
│     • r - g positive (unfavorable) for {:.0f}/{:.0f} years                        │
│                                                                             │
│  2. UNIT ROOT TESTS:                                                        │
│     • Debt_GDP, Interest_Rate, Primary_Deficit are I(1)                     │
│     • GDP_Growth is I(0)                                                    │
│                                                                             │
│  3. COINTEGRATION ANALYSIS:                                                 │
""".format(df['Debt_GDP'].mean(), df['Debt_GDP'].min(), df['Debt_GDP'].max(),
           df['GDP_Growth'].mean(), df['GDP_Growth'].min(), df['GDP_Growth'].max(),
           (df['r_minus_g'] > 0).sum(), len(df)))

# Check cointegration conclusion from earlier
cointegration_found = any(johansen_result.lr1[i] > johansen_result.cvt[i, 1] for i in range(len(johansen_result.eig)))
if cointegration_found:
    print("     • Johansen test confirms cointegrating relationships")
    print("     • Long-run equilibrium exists among variables")
else:
    print("     • No strong evidence of cointegration")

print("""
│                                                                             │
│  4. MODEL 1 - DEBT DETERMINANTS:                                           │
""")
print(f"     • GDP Growth coefficient: {model1.params['GDP_Growth']:.4f} (p={model1.pvalues['GDP_Growth']:.4f})")
print(f"     • Interest Rate coefficient: {model1.params['Interest_Rate']:.4f} (p={model1.pvalues['Interest_Rate']:.4f})")
print(f"     • Primary Deficit coefficient: {model1.params['Primary_Deficit']:.4f} (p={model1.pvalues['Primary_Deficit']:.4f})")
print(f"     • R²: {model1.rsquared:.4f}")

print("""
│                                                                             │
│  5. MODEL 2 - BOHN SUSTAINABILITY TEST:                                     │
""")
print(f"     • β (Debt response coefficient): {beta_debt:.4f}")
print(f"     • p-value: {beta_pvalue:.4f}")
if beta_debt > 0 and beta_pvalue < 0.05:
    print("     • CONCLUSION: FISCALLY SUSTAINABLE ✓")
elif beta_debt > 0 and beta_pvalue < 0.10:
    print("     • CONCLUSION: WEAK EVIDENCE OF SUSTAINABILITY ⚠")
else:
    print("     • CONCLUSION: SUSTAINABILITY NOT CONFIRMED ✗")

print("""
│                                                                             │
│  6. DIAGNOSTIC TESTS:                                                       │
│     • Durbin-Watson: {:.4f}                                                    │
│     • BG LM Test p-value: {:.4f}                                               │
│     • White Test p-value: {:.4f}                                               │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""".format(dw_stat_model2, bg_test_model2[1], white_test_model2[1]))

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)


SECTION 8: SUMMARY OF FINDINGS

┌─────────────────────────────────────────────────────────────────────────────┐
│                    FISCAL SUSTAINABILITY ANALYSIS SUMMARY                   │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  1. DESCRIPTIVE ANALYSIS:                                                   │
│     • Average Debt-GDP ratio: 71.4% (Range: 65.2% - 89.0%)                │
│     • Average GDP Growth: 6.3% (Range: -6.6% - 10.3%)                   │
│     • r - g positive (unfavorable) for 28/34 years                        │
│                                                                             │
│  2. UNIT ROOT TESTS:                                                        │
│     • Debt_GDP, Interest_Rate, Primary_Deficit are I(1)                     │
│     • GDP_Growth is I(0)                                                    │
│              